In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import sys

# Lecture 14: Import MNIST Images
1. Convert MNIST image files into a Tensor of 4-Dimensions: Number of Images, Height, Weight, Color Channels
2. Set train and test data

In [23]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root=r"./Datasets/MNIST", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root=r"./Datasets/MNIST", train=False, download=True, transform=transform)
train_data, test_data

(Dataset MNIST
     Number of datapoints: 60000
     Root location: ./Datasets/MNIST
     Split: Train
     StandardTransform
 Transform: ToTensor(),
 Dataset MNIST
     Number of datapoints: 10000
     Root location: ./Datasets/MNIST
     Split: Test
     StandardTransform
 Transform: ToTensor())

# Lecture 15: Convolutional and Pooling Layers
1. Create a small batch size for images. Let's say 10.
2. Define CNN Model by describing convolutional layers.
3. Grab 1 MNIST record/image.
4. Pass Through 2 Convolutional Layers.
5. Pass Through Pooling Layer.

## Quick Note for Convolutional and Pooling Layers:
1. Out put size equation:<center>$O=\frac{W-K+2{\times}P}{S}+1$</center>
    * O: Output Size
    * W: input Volume
    * K: Kernel Size
    * P: Padding
    * S: Stride
2. If padding is not set, we will lose image boundary pixels.
3. Max pooling will round down if output size has remainder.

In [24]:
# Step 1
train_loader = DataLoader(train_data, batch_size=50, shuffle=True)
test_loader = DataLoader(test_data, batch_size=50, shuffle=False)


# Step 2
conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=3, stride=1)
conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, stride=1)


# Step 3
for idx, (X_train, y_train) in enumerate(train_data):
    break

X = X_train.reshape(1, 1, 28, 28)
print(X.shape)


# Step 4
X = F.relu(conv1(X))
print(X.shape)

X = F.relu(conv2(X))
print(X.shape)


# Step 5
X = F.max_pool2d(X, kernel_size=2, stride=2)
print(X.shape)



torch.Size([1, 1, 28, 28])
torch.Size([1, 6, 26, 26])
torch.Size([1, 16, 24, 24])
torch.Size([1, 16, 12, 12])


# Lecture 16: Convolutional Neural Network Model
1. Create a Model Class:
    * Create Convolutional Layers and Pooling Layers.
    * Connect all Layers.
2. Create an Instance of our Model
3. Create Loss Function and Optimizor

In [25]:
# Step 1
class ConvolutionalNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.LazyConv2d(out_channels=6, kernel_size=3, stride=1)
        self.conv2 = nn.LazyConv2d(out_channels=16, kernel_size=3, stride=1)

        self.fc1 = nn.LazyLinear(out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=90)
        self.fc3 = nn.Linear(in_features=90, out_features=10)

    def forward(self, X):
        X = F.relu(conv1(X))
        X = F.relu(conv2(X))
        X = F.max_pool2d(X, kernel_size=2, stride=2)
        X = X.flatten(start_dim=1)

        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = F.log_softmax(X, dim=1)

        return X


# Step 2
torch.manual_seed(42)
model = ConvolutionalNetwork()


# Step 3
criterion = nn.CrossEntropyLoss()
optimizor = torch.optim.Adam(model.parameters(), lr=0.001)

# Lecture 17: Train and Test CNN Network
1. Create Variables to Track Things
2.

In [26]:
epochs = 5
total_train_losses = []
total_test_losses = []
total_train_corrects = []
total_test_corrects = []


for idx in range(epochs):
    train_corrects = []
    test_corrects = []

    pbar = tqdm(train_loader, desc=f"Epoch {idx+1}, Batch Progress",unit="batch", file=sys.stdout)
    for batch, (X_train, y_train) in enumerate(pbar):
        y_pred = model.forward(X_train)
        loss = criterion(y_pred, y_train)


        optimizor.zero_grad()
        loss.backward()
        optimizor.step()

        pbar.set_postfix({"loss": f"{loss:.4f}"})

        # batch += 1

Epoch 5, Batch Progress: 100%|██████████| 1200/1200 [00:25<00:00, 46.63batch/s, loss=0.0274]
